# nano Beatrix — sixteen bound pairs with the records' keys held: capacity, lane 2 of 2

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbstractEyes/geolip-nano-beatrix-trainer/blob/main/colab/nano_capacity_r16_lane2.ipynb)

**The question**: Sixteen bound pairs in the told head keep a million competing keys apart on most models; the few wrong looks left come from other rows' records whose key pairs run longer than the true record's. The G4 notebook ([`nano_rec16.ipynb`](https://colab.research.google.com/github/AbstractEyes/geolip-nano-beatrix-trainer/blob/main/colab/nano_rec16.ipynb)) trains six sixteen-pair models with the key's pairs held at one length at the records' end-markers only (the index entries keep their own lengths). How many keys does each keep apart?

**This notebook** trains nothing. As each of the six lands on the Hub it reads the model's capacity the way the sixteen-pair models were read (the same 192 rows a model, half with key-hard keys; the same bank of up to 10^6 competing keys, each a stored end-marker's content turned by a fresh key's phases): the leak and the look error at N = 10^3, 4,096 (a window's worth of keyed end-markers), 10^4, 10^5 and 10^6, three ways on the same competitors (as read; with every competitor's pair lengths set to its entry's true record's; with every true score set to the model's median), beside the random-phase moment at the model's own bound pairs J and margin s a pair, N I0(s)^J / (exp(J s) + N I0(s)^J), and each look error's strongest competitor.

**The lanes**: two lanes on two CPU runtimes beside the G4: every lane walks the six in the card's order, waits until a model has landed whole on the Hub (its weights, its phases, its record), and claims it before reading it, by a small file under `bed/preflight/capacity_r16_claims/` written only if the repo has not moved since the lane's read of it, so no two lanes read one model; each record ships to `bed/reports/capacity_tail/` as it lands. A record does not depend on its lane.

**The cells**: C1 pins the trainer and holds the decision block and the lane's name; C2 writes the held-out documents (the rows' filler), lists the models already on the Hub and probes the push path; C3 is the lane; C4 renders the tables of the records this lane read.

**Before C1**: Runtime > Change runtime type > CPU. The notebook's `HF_TOKEN` secret (write access to [AbstractPhil/alephllm-nano-beatrix-training](https://huggingface.co/AbstractPhil/alephllm-nano-beatrix-training)) with notebook access on: a lane claims before it reads, so a tokenless lane reads nothing. Run C1, then Runtime > Restart session and run all. A file named `STOP` in `/content/out` starts no new model; a restarted session takes back its own unfinished claim. The code and its tests: [AbstractEyes/geolip-nano-beatrix-trainer](https://github.com/AbstractEyes/geolip-nano-beatrix-trainer) (`geolip/nano_beatrix/capacity.py`, `tests/test_capacity.py`, `tests/test_records_hold.py`).

In [ ]:
# C1 — pinned installs + THE DECISION BLOCK (a CPU runtime). RESTART the session after the first run of this cell (Runtime > Restart session and run all).
import os, sys
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'   # a progress bar is not allowed to cost a session
%pip install -q "geolip-nano-beatrix-trainer[corpus] @ git+https://github.com/AbstractEyes/geolip-nano-beatrix-trainer@v0.1.88" "amoe-lora @ git+https://github.com/AbstractEyes/amoe-lora@v0.2.12"
import torch, amoe
import geolip.nano_beatrix as N
assert tuple(int(x) for x in N.__version__.split('.')) >= (0, 1, 88), (
    'RESTART REQUIRED: the runtime still holds an older package — Runtime > Restart session, then rerun from C1')
assert amoe.__version__ == '0.2.12', 'the arm module of record is amoe-lora 0.2.12'
torch.set_num_threads(os.cpu_count() or 2)
print('geolip.nano_beatrix', N.__version__, '| amoe', amoe.__version__, '| torch', torch.__version__, '| cpu threads', torch.get_num_threads(),
      '| card', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none (the read runs on the CPU)')

def _token():
    try:
        from google.colab import userdata
        return userdata.get('HF_TOKEN')
    except Exception:
        return os.environ.get('HF_TOKEN')
HF_TOKEN = _token()
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN            # the runner and the shipper read it from the environment; the value is never printed

# ============ THE DECISION BLOCK — every setting from the plan, with its source beside it ============
OUT = '/content/out'                                   # the session folder: the held-out documents, the records, a model's weights while it is read
REPO = 'AbstractPhil/alephllm-nano-beatrix-training'  # the models' weights are read from here; every record ships here
DEST = 'bed/reports/capacity_tail'                     # the records' folder on the Hub
R = 192                                                # rows a model, half key-hard: the sixteen-pair models' read
SEED = 1                                               # the rows' generator and the bank's: the sixteen-pair models' read
from geolip.nano_beatrix import capacity as _cp
NS = _cp.TAIL_NS                                       # the competitor counts: 10^3, 4,096 (a window's keyed end-markers), 10^4, 10^5, 10^6
TAIL = '_L24000_ctx4096_frdyn_cap_mode_sz32-128-256-512_r4_n30_o0_askany_gag50'
FAM = {'L16R': '_cp4_gc48_kh50_va1_pt10b4h0_krt16r218_ahdigit1_rec16'}   # 48 records, the key-hard rows, sixteen bound pairs, the records held
LANE = 'lane2'                                      # this lane's name: its claims carry it
CORES = [(f'{f} s{s}', f'alhena-both_s{s}{TAIL}{FAM[f]}') for s in range(1, 7) for f in ('L16R',)]   # the card's order
WAIT_S, DEADLINE_H = 120, 9.0                          # a lane waiting on the card reads the Hub every 2 min, at most 9 h
print('decision block:', LANE, '|', len(CORES), 'models |', R, 'rows a model, seed', SEED, '| N', NS, '| ships to',
      REPO + '/' + DEST if HF_TOKEN else 'nothing (tokenless)')

In [ ]:
# C2 (P) — PREFLIGHT: the held-out documents (the rows' filler), the models already on the Hub, the push probe. Nothing is read yet.
import json, os, time
from huggingface_hub import HfApi
from geolip.nano_beatrix import bed_corpus
HELD = bed_corpus.write_heldout(f'{OUT}/bed')         # the first 2,048 documents of fineweb-edu sample-10BT (the beds' held-out head)
print('held-out documents', HELD, os.path.getsize(HELD) / 1e6, 'MB')
api = HfApi(token=HF_TOKEN or False)
FILES = set(api.list_repo_files(REPO))
landed = [lab for lab, tag in CORES if f'bed/weights/{tag}/key_rot.safetensors' in FILES]   # the rest are still training
done_hub = [lab for lab, tag in CORES if f'{DEST}/{_cp.tail_name(tag, R, SEED)}' in FILES]
print('models on the Hub with their phases:', landed or 'none yet (the lane waits for the card)', '| records already shipped (done before):', done_hub or 'none')
PUSH_OK = False
if HF_TOKEN:
    path = f'bed/preflight/push_probe_capacity_r16_{LANE}.json'
    body = json.dumps({'utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()), 'cpu_threads': torch.get_num_threads(),
                       'models': [lab for lab, _ in CORES], 'rows': R}).encode()
    api.upload_file(path_or_fileobj=body, path_in_repo=path, repo_id=REPO, commit_message=f'nano capacity r16: push probe ({LANE})')
    PUSH_OK = path in set(api.list_repo_files(REPO))
    print('push probe', 'READ BACK OK' if PUSH_OK else 'NOT VISIBLE after upload', path)
else:
    print('TOKENLESS: no probe written; the lane reads and ships nothing')

In [ ]:
# C3 (R) — THE LANE: the six models in the card's order, one at a time, each once it has landed whole on the Hub. Before each model the Hub is read at one commit: a
# model with its record there (or in OUT/capacity) is done; a model with another lane's claim younger than FRESH_S is taken; the first
# model with neither is claimed (a small file under CLAIMS, written only if the repo has not moved since that read - else the read repeats),
# read, and its record shipped as it lands (a background thread with retries; the local record is the done mark). A STOP file in OUT starts
# no new model. The lane ends when no model is free. The job's size, the time spent and the time left print after every model.
import json, os, random, shutil, threading, time
from huggingface_hub import snapshot_download
from geolip.nano_beatrix import capacity as cp, rotary_scale as rs
assert HF_TOKEN and PUSH_OK, 'a lane claims before it reads: it needs the token and the push probe read back (C2)'
held = rs.load_heldout(HELD)
REC = f'{OUT}/capacity_tail'
CLAIMS = 'bed/preflight/capacity_r16_claims'
FRESH_S = 40 * 60                                      # a claim this old with no record is a stopped lane's: the model is free again
os.makedirs(REC, exist_ok=True)
threads, failed = [], []

def _ship(path):
    for i in range(6):
        try:
            api.upload_file(path_or_fileobj=path, path_in_repo=f'{DEST}/{os.path.basename(path)}', repo_id=REPO,
                            commit_message=f'nano capacity r16: {os.path.basename(path)[:90]}')
            return
        except Exception as e:
            print('ship retry', i + 1, os.path.basename(path)[:60], type(e).__name__)
            time.sleep(min(300, 10 * 2 ** i))
    failed.append(path)

def ship(path):
    t = threading.Thread(target=_ship, args=(path,))
    t.start()
    threads.append(t)

def snap():
    # the repo's head commit and its files, read at that one commit
    sha = api.repo_info(REPO).sha
    return sha, set(api.list_repo_files(REPO, revision=sha))

def done(tag, files):
    name = cp.tail_name(tag, R, SEED)
    return f'{DEST}/{name}' in files or os.path.exists(f'{REC}/{name}')

def claims(label, files):
    # the fresh claims on a model, [(lane, utc seconds)], from the claim files' names: <label>__<lane>__<utc seconds>.json
    pre = f"{CLAIMS}/{label.replace(' ', '_')}__"
    out = []
    for f in files:
        if f.startswith(pre) and f.endswith('.json'):
            lane, utc = f[len(pre):-len('.json')].split('__')
            if time.time() - int(utc) < FRESH_S:
                out.append((lane, int(utc)))
    return out

def ready(tag, files):
    # a model has landed whole on the Hub: its weights, its phases, its scale.json and its run's record
    return all(f in files for f in (f'bed/weights/{tag}/model.safetensors', f'bed/weights/{tag}/key_rot.safetensors',
                                    f'bed/weights/{tag}/scale.json', f'bed/reports/bed_{tag}.json'))

def pick():
    # -> (label, tag) claimed by this lane, 'WAIT' when the models left are still training, or None when no model is left
    for attempt in range(40):
        sha, files = snap()
        for label, tag in CORES:                       # this lane's own claim on an unfinished model (a restarted session) is still its
            if not done(tag, files) and any(lane == LANE for lane, _ in claims(label, files)):
                return label, tag
        left = [(label, tag) for label, tag in CORES if not done(tag, files) and not claims(label, files)]
        if not left:
            return None
        free = [(label, tag) for label, tag in left if ready(tag, files)]
        if not free:
            return 'WAIT'                               # every model left for this lane is still on the card
        label, tag = free[0]
        path = f"{CLAIMS}/{label.replace(' ', '_')}__{LANE}__{int(time.time())}.json"
        try:
            api.upload_file(path_or_fileobj=json.dumps({'lane': LANE, 'model': tag}).encode(), path_in_repo=path, repo_id=REPO,
                            commit_message=f'nano capacity r16: {LANE} claims {label}', parent_commit=sha)
            return label, tag
        except Exception as e:                          # the repo moved since the read (another lane's claim or record) or the call failed:
            print('claim retry', attempt + 1, label, type(e).__name__)   # read again (a claim that landed anyway is found as this lane's)
            time.sleep(random.uniform(3, 12))
    raise RuntimeError('no claim after 40 tries')

_, FILES = snap()
for f in sorted(os.listdir(REC)):                     # local records not on the Hub: shipped first
    if f.endswith('.json') and f'{DEST}/{f}' not in FILES:
        ship(f'{REC}/{f}')
t0, read_n, recs, waits = time.time(), 0, {}, 0
while True:
    if os.path.exists(f'{OUT}/STOP'):
        print('STOP present: no new model starts')
        break
    got = pick()
    if got is None:
        print('no model is left: the lane ends')
        break
    if got == 'WAIT':
        if time.time() - t0 > DEADLINE_H * 3600:
            print(f'no model landed for this lane in {DEADLINE_H} h: the lane ends')
            break
        waits += 1
        if waits % 5 == 1:
            print(f"waiting for the card: {time.strftime('%H:%M:%S', time.gmtime())} UTC, {read_n} read by this lane, "
                  f"{(time.time() - t0) / 60:.0f} min since the lane started")
        time.sleep(WAIT_S)
        continue
    label, tag = got
    local = f'{REC}/{cp.tail_name(tag, R, SEED)}'
    t1 = time.time()
    snapshot_download(REPO, allow_patterns=[f'bed/weights/{tag}/*'], local_dir=f'{OUT}/w', token=HF_TOKEN)
    cp.run_tail(f'{OUT}/w/bed/weights/{tag}', held, REC, R=R, seed=SEED, ns=NS, log=print)
    ship(local)
    shutil.rmtree(f'{OUT}/w/bed/weights/{tag}', ignore_errors=True)
    read_n += 1
    rec = recs[label] = json.load(open(local, encoding='utf-8'))
    b, m_, e_ = rec['base'], rec['M'], rec['E']
    _, files = snap()
    open_n = sum(not done(tag_, files) for _, tag_ in CORES)
    spent = time.time() - t0
    print(f"{label} [{LANE}]: s {rec['s_core']:.2f} | look error at 10^6: base {b['1000000']['look_error']:.4f}, M (the competitors' "
          f"lengths = the true record's) {m_['1000000']['look_error']:.4f}, E (the true score = the median) {e_['1000000']['look_error']:.4f} | "
          f"base at 4,096 {b['4096']['look_error']:.5f} | {time.time() - t1:.0f} s | this lane: {read_n} read in {spent / 60:.1f} min | "
          f"{open_n} of {len(CORES)} models without a record, ~{open_n * spent / read_n / 2 / 60:.0f} min left at 2 lanes (the card's pace bounds it)")
for t in threads:
    t.join()
_, FILES = snap()
print('this lane read', read_n, '| UNSHIPPED:', [f for f in sorted(os.listdir(REC)) if f.endswith('.json') and f'{DEST}/{f}' not in FILES])

In [ ]:
# C4 — THE TABLES (the records in OUT/capacity_tail): per model the look error and the mean leak at 10^6 three ways, the base look error at
# a window's worth of keys, the pooled records' pair lengths over the true records' (the largest pair's 99th percentile), and where the base
# look errors' strongest competitors come from.
from IPython.display import Markdown, display
rows = ['| model | s | look error 10^6: base / M / E | mean leak 10^6: base / M / E | base look error 4,096 | pooled records: length p99 / true | '
        'strongest competitor: own record / its row / another row / an index entry |', '|---|---|---|---|---|---|---|']
for label, tag in CORES:
    if label not in recs:
        continue
    r = recs[label]
    le = ' / '.join(f"{r[c]['1000000']['look_error']:.4f}" for c in ('base', 'M', 'E'))
    ml = ' / '.join(f"{r[c]['1000000']['mean']:.3f}" for c in ('base', 'M', 'E'))
    src = ' / '.join(str(v) for v in r['top_competitor']['source'].values())
    rows.append(f"| {label} | {r['s_core']:.2f} | {le} | {ml} | {r['base']['4096']['look_error']:.5f} | "
                f"{max(r['pool_length_ratio']['records']['p99']):.2f} | {src} |")
display(Markdown('\n'.join(rows)))